In [1]:
import os
import requests
from flask import Flask, request, abort
from pyngrok import ngrok
from google import genai


PORT = 5051

NGROK_AUTHTOKEN = "3EO3VV2PaChC668gAUNtfMWlSAN_2zPGKHv6S3tF64dVC2tpm"
LINE_CHANNEL_ACCESS_TOKEN = "jGLSUVlmTGB3AmOQ2I63e0CrHi8ir/zTpzPXcD/M45hRlHslU0NCjGDoAkfl99jIMNIVbVJo/MQYU6I31grF168BgZffL1UbU5PyVueBpNn+MG84ud1a81LSPNQ5gqjZyO5Y203Ht+qrYlsPvVt03AdB04t89/1O/w1cDnyilFU="
LINE_CHANNEL_SECRET = "eab180b7664618e531419af01b8bd690"


GEMINI_API_KEY = "AQ.Ab8RN6KJxdDFhrl4F3uy4iPktmFIWG8Z6qs81aZptQQC_uT_mg"
# ====================================================================

from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import MessageEvent, TextMessageContent

# 初始化 Gemini Client
client = genai.Client(api_key=GEMINI_API_KEY)

app = Flask(__name__)
configuration = Configuration(access_token=LINE_CHANNEL_ACCESS_TOKEN)
handler = WebhookHandler(LINE_CHANNEL_SECRET)

def stateless_query(payload):
    """呼叫 Gemini 2.5 Flash 模型獲取回答"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=payload
    )
    return response.text

def update_line_webhook(webhook_url):
    """自動更新 LINE 後台的 Webhook 網址"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {LINE_CHANNEL_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }
    data = {"endpoint": webhook_url}
    response = requests.put(url, headers=headers, json=data)
    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ Webhook 更新失敗：{response.status_code} - {response.text}")
        return False

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers.get('X-Line-Signature', '')
    body = request.get_data(as_text=True)
    print("📥 BODY: ", body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        abort(400)
    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 🎯 0701 作業核心邏輯判斷
        if text.startswith('AI '):
            prompt = text[3:]  # 去掉前三個字元 "AI "，剩下的拿去問 Gemini
            print(f"🤖 正在詢問 Gemini AI: {prompt}")
            reply_text = stateless_query(prompt)

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )
        else:
            # 如果一般聊天，依然重複彈出兩次
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=text),
                        TextMessage(text=text)
                    ]
                )
            )

if __name__ == "__main__":
    try:
        ngrok.kill()
    except:
        pass

    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunnel = ngrok.connect(PORT, name="linebot_tunnel")
    webhook_url = tunnel.public_url
    print(f"🚀 Ngrok URL: {webhook_url}")

    update_line_webhook(webhook_url)

    print(f"📡 正在本地啟動 Flask 伺服器，使用 Port: {PORT}...")
    app.run(port=PORT)

ModuleNotFoundError: No module named 'pyngrok'